In [1]:
!pip install optuna

In [2]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

def df_Data_Cleansing(df):
    cat_cols = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().astype('category')

    #14
    df['short_sleep_flag'] = (df['sleep_duration'] < 6.0).astype(int)

    #15
    df["exercise_strength"] = df['calorie_expenditure'] / ((df['exercise_duration'] + 0.1) * df['bmi'])

    #16
    df['healthy_score'] = (
        df['sleep_duration'].between(7, 8).astype(int) +
        (df['bmi'] <= 19).astype(int) +
        (df['exercise_duration'] >= 40).astype(int) +
        (df['stress_level'] == 'low').astype(int) +
        (df['sleep_quality'] == 'good').astype(int) +
        (df['physical_activity_level'] == 'active').astype(int)
    )

    #17
    df['unhealthy_score'] = (
        df['sleep_duration'].between(5, 6).astype(int) +
        (df['bmi'] >= 27).astype(int) +
        (df['calorie_expenditure'] <= 1400).astype(int) +
        (df['step_count'] <= 5000).astype(int) +
        (df['exercise_duration'] <= 30).astype(int) +
        (df['stress_level'] == 'high').astype(int) +
        (df['sleep_quality'] == 'poor').astype(int) +
        df['physical_activity_level'].isin(['sedentary', 'moderate']).astype(int) +
        (df['smoking_alcohol'] == 'yes').astype(int)
    )

    #18
    stress_mapping = {'low': 1, 'medium': 2, 'high': 3}
    df['stress_level_num'] = df['stress_level'].map(stress_mapping).astype(float)
    df['stress_per_sleep'] = df['stress_level_num'] / df['sleep_duration']

    # 不要な特徴量をまとめて削除
    drop_cols = [
        'stress_level_num',  # 計算用の一時データ（重複）
        'diet_type',
        'gender',
        'smoking_alcohol',
    ]
    df = df.drop(columns=drop_cols, errors='ignore')

    return df



# データの読み込み（パスはColabの環境に合わせて変更してください）
df = df_Data_Cleansing(pd.read_csv('/content/train.csv', encoding="utf-8"))

X = df.drop(['id', 'health_condition'], axis=1)
y = df['health_condition']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Optunaの目的関数
def objective(trial):
    # 探索するパラメータの範囲を指定
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_error',
        'verbosity': -1,
        'class_weight': 'balanced',
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, valid_idx in cv.split(X, y_encoded):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y_encoded[train_idx], y_encoded[valid_idx]

        model = lgb.LGBMClassifier(**params, n_estimators=1000)

        # Early Stoppingを設定して過学習を防ぐ
        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        preds = model.predict(X_valid)
        score = balanced_accuracy_score(y_valid, preds)
        scores.append(score)

    return np.mean(scores)

print("Optunaによるパラメータ探索を開始します...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30) # まずは30回お試しで探索

print("=========================")
print(f"ベストスコア (Balanced Accuracy): {study.best_value:.5f}")
print("ベストパラメータ:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")
print("=========================")

[I 2026-07-20 13:29:21,180] A new study created in memory with name: no-name-ee6c6059-fb64-4d41-92bf-adc47fec2536


Optunaによるパラメータ探索を開始します...


[I 2026-07-20 13:30:20,549] Trial 0 finished with value: 0.948035419557187 and parameters: {'learning_rate': 0.009245993903419665, 'num_leaves': 36, 'max_depth': 8, 'min_child_samples': 94, 'subsample': 0.9561128598360435, 'colsample_bytree': 0.7887267627223775}. Best is trial 0 with value: 0.948035419557187.
[I 2026-07-20 13:42:09,131] Trial 1 finished with value: 0.9496441271134515 and parameters: {'learning_rate': 0.04242643904153932, 'num_leaves': 27, 'max_depth': 4, 'min_child_samples': 34, 'subsample': 0.988221876154356, 'colsample_bytree': 0.6419394488306607}. Best is trial 1 with value: 0.9496441271134515.
[I 2026-07-20 13:50:52,024] Trial 2 finished with value: 0.9491142594959087 and parameters: {'learning_rate': 0.026103304463160424, 'num_leaves': 72, 'max_depth': 7, 'min_child_samples': 66, 'subsample': 0.9059967448542741, 'colsample_bytree': 0.7546385485853709}. Best is trial 1 with value: 0.9496441271134515.
[I 2026-07-20 13:52:21,798] Trial 3 finished with value: 0.947807

ベストスコア (Balanced Accuracy): 0.94970
ベストパラメータ:
    learning_rate: 0.035713560370948735
    num_leaves: 86
    max_depth: 4
    min_child_samples: 10
    subsample: 0.8169256013051673
    colsample_bytree: 0.6381386148722236
